# Compile for hardware constraints

This is Notebook 3 for the SC26 tutorial.
We will compile a circuit for two hardware models, inspect native gates and routing, and check the logical results by local simulation.

Start with a fresh kernel and run the cells in order.
This notebook uses the same Python 3.15 environment as the rest of the tutorial.

In [ ]:
from matplotlib import pyplot as plt
from qiskit import QuantumCircuit
from qiskit.visualization import plot_distribution

from mqt.core.mlir import CompilerTarget, OutputFormat, compile_program, sample

shots = 4096
seed = 17

## Predict the logical result

After the first two controlled-$X$ gates, we have a GHZ state.
The final controlled-$X$ flips qubit 2 when qubit 0 is 1, giving $(|000\rangle + |011\rangle)/\sqrt{2}$ in basis order $|q_2q_1q_0\rangle$.

In [ ]:
source = QuantumCircuit(3)
source.h(0)
source.cx(0, 1)
source.cx(1, 2)
source.cx(0, 2)
source.measure_all()
source.draw("mpl")

Let us check the expected logical output before introducing a compiler target:

In [ ]:
logical_counts = sample(source, shots=shots, seed=seed)
plot_distribution(logical_counts, title="Logical output distribution")

## Describe two compiler targets

We describe the hardware constraints with a `CompilerTarget`, which lists sites, connectivity, and native operations.
Our two three-site models support the same gates: $R_z$, $R_y$, and controlled-$X$, plus measurement.
Since $H$ is not native, the compiler will have to synthesize it from rotations.

In [ ]:
native_operations = CompilerTarget.NativeOperations([
    CompilerTarget.OperationCapability("rz", arity=1, num_parameters=1),
    CompilerTarget.OperationCapability("ry", arity=1, num_parameters=1),
    CompilerTarget.OperationCapability("cx", arity=2, num_parameters=0),
    CompilerTarget.OperationCapability("measure", arity=1, num_parameters=0),
])

all_to_all_target = CompilerTarget(
    3,
    connectivity=CompilerTarget.Connectivity.all_to_all(),
    native_operations=native_operations,
)

line_edges = [(0, 1), (1, 2)]
line_target = CompilerTarget(
    3,
    connectivity=CompilerTarget.Connectivity(line_edges),
    native_operations=native_operations,
)

In these operation definitions, `arity` gives the number of qubits and `num_parameters` gives the number of gate parameters.
The two targets differ only in their connectivity:

In [ ]:
positions = [(0, 0), (1, 0), (2, 0)]
fig, axes = plt.subplots(1, 2, figsize=(9, 2.4))
for ax, (name, edges) in zip(axes, [("All-to-all", [(0, 1), (1, 2), (0, 2)]), ("Line", line_edges)], strict=True):
    for left, right in edges:
        if (left, right) == (0, 2):
            ax.annotate(
                "",
                xy=positions[right],
                xytext=positions[left],
                arrowprops={
                    "arrowstyle": "-",
                    "connectionstyle": "arc3,rad=-0.45",
                    "color": "#0065bd",
                },
            )
        else:
            ax.plot([left, right], [0, 0], color="#0065bd", linewidth=2)
    ax.scatter([0, 1, 2], [0, 0, 0], s=650, color="#0065bd", zorder=3)
    for site, (x, y) in enumerate(positions):
        ax.text(x, y, str(site), ha="center", va="center", color="white", fontsize=12)
    ax.set(title=name, xlim=(-0.4, 2.4), ylim=(-0.3, 0.8))
    ax.axis("off")
fig.suptitle("Physical sites and available connections")
fig.tight_layout()

## Compile, then inspect the native circuits

The compiler uses **placement** to assign qubits to sites and **routing** to bring interacting quantum states together.
**Native synthesis** rewrites the remaining gates using operations the target supports.

:::{admonition} Your turn
:class: tip

Before compiling, predict which target needs more controlled-$X$ gates:
:::

### All-to-all target

In [ ]:
all_to_all_program = compile_program(source, target=all_to_all_target, output=OutputFormat.QC)
all_to_all_program.to_qiskit(copy=True).draw("mpl", fold=12)

### Line target

In [ ]:
line_program = compile_program(source, target=line_target, output=OutputFormat.QC)
line_program.to_qiskit(copy=True).draw("mpl", fold=12)

The drawings now show physical sites.
Since routing exchanges are decomposed into native gates, we do not see explicit $\mathrm{SWAP}$ symbols.
Let us compare the gate counts:

In [ ]:
all_to_all_gates = all_to_all_program.gate_counts()
line_gates = line_program.gate_counts()

print("Target       RZ   RY   Controlled-X")
print(
    f"All-to-all {all_to_all_gates.get('rz', 0):4} {all_to_all_gates.get('ry', 0):4} {all_to_all_gates.get('cx', 0):14}"
)
print(f"Line       {line_gates.get('rz', 0):4} {line_gates.get('ry', 0):4} {line_gates.get('cx', 0):14}")

## Compare the logical results

Routing changes physical qubit locations, while measurements keep their original classical-bit assignments.
Let us compare the compiled programs with the original circuit:

In [ ]:
all_to_all_counts = sample(all_to_all_program, shots=shots, seed=seed)
line_counts = sample(line_program, shots=shots, seed=seed)
plot_distribution([logical_counts, all_to_all_counts, line_counts], legend=["Logical", "All-to-all", "Line"])

Although the line topology requires extra gates, both targets give the expected logical distribution.

:::{admonition} Your turn
:class: tip

Explain why the line topology needs more native gates and why its logical outputs still agree with the all-to-all target.
:::

## Checkpoint: Return to the slides

We return to the slides here.
The experiments below are optional.

## Experiment: Remove the routing constraint

Let us replace the line topology with a triangle, connecting every pair of sites.

:::{admonition} Your turn
:class: tip

Before compiling, predict whether any extra routing gates will be needed:
:::

In [ ]:
triangle = CompilerTarget(
    3,
    connectivity=CompilerTarget.Connectivity([*line_edges, (0, 2)]),
    native_operations=native_operations,
)
triangle_program = compile_program(source, target=triangle, output=OutputFormat.QC)
print(triangle_program.gate_counts())

:::{dropdown} Explain the result

With every pair connected, the triangle needs no routing, though $H$ still has to be synthesized from native rotations.
:::

## Experiment: Provide too few sites

A two-site target has too little space for our three-qubit program, so this compilation should fail:

In [ ]:
too_small = CompilerTarget(
    2,
    connectivity=CompilerTarget.Connectivity.all_to_all(),
    native_operations=native_operations,
)
try:
    compile_program(source, target=too_small, output=OutputFormat.QC)
except RuntimeError as error:
    print(f"Expected compilation failure: {str(error).splitlines()[0]}")

## Next steps

After the HPC integration slides, we will use `qdmi_execution.ipynb` to run a compiled program through QDMI.

:::{admonition} Read further
:class: seealso

For other targets and options, see the [target compilation guide](https://mqt.readthedocs.io/projects/core/en/stable/mlir/target_compilation.html).
:::